In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
import pandas as pd
import numpy as np
from sklearn.metrics import accuracy_score, classification_report
import torch

c:\Users\choyb\Documents\DOST\react-django\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
# {0: "Negative", 1: "Positive", 2: "Neutral"}
model_name = "dost-asti/RoBERTa-tl-sentiment-analysis"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=3)

Loading weights: 100%|██████████| 201/201 [00:00<00:00, 21324.81it/s]


In [ ]:
from datasets import load_dataset

#0 → Negative, 1 → Neutral, 2 → Positive
ds = load_dataset("jjjardev/tagasenti")
ds

DatasetDict({
    train: Dataset({
        features: ['sentence', 'label'],
        num_rows: 35686
    })
})

In [ ]:
#Convert the dataset to a pandas dataframe for data manipulation
df = ds["train"].to_pandas()
df.head(5)

,sentence,label
0,i'm disappointed pero i expected q naman haha ...,0
1,Hindi kapanipaniwala ang mga dahilan na ibinig...,0
2,"Si Duterte, ala-Hitler; Walang alam si Grace; ...",0
3,Napakagandang farewell album ito ng The Beatle...,2
4,di ko na gustuhan texture nya amoy chemical na...,0


In [ ]:
# # Match the dataset labels to the dost-asti model
dataset_to_dost_asti = {0:0, 1:2, 2:1}
dataset_to_dost_asti_string = {0: "Negative", 1: "Positive", 2: "Neutral"}
df["label"] = df["label"].map(dataset_to_dost_asti)
df.head(5)

,sentence,label
0,i'm disappointed pero i expected q naman haha ...,0
1,Hindi kapanipaniwala ang mga dahilan na ibinig...,0
2,"Si Duterte, ala-Hitler; Walang alam si Grace; ...",0
3,Napakagandang farewell album ito ng The Beatle...,1
4,di ko na gustuhan texture nya amoy chemical na...,0


In [ ]:
# Copy the dataset with n rows for quick testing
sample_df = df.sample(n=500, random_state=42).copy()
len(sample_df)

500

In [ ]:
vc =sample_df["label"].map(dataset_to_dost_asti_string).value_counts()
vc

label
Negative    182
Positive    159
Neutral     159
Name: count, dtype: int64

In [ ]:
# Tokenize the text
inputs = tokenizer(sample_df["sentence"].to_list(), return_tensors="pt", truncation=True, max_length=512, padding=True)

In [ ]:

with torch.no_grad():
   # Covert the inputs to logits
   output = model(**inputs)

   # Put the logits to a softmax function to determine the probabilities for each sentiment
   # dim=1 means turn each row of logits into probabilities that sum to 1
   probabilities = torch.softmax(output.logits, dim=-1).detach().cpu().numpy()

   # Get the highest probability   
   predicted_sentiment = torch.argmax(output.logits, dim=-1)

In [ ]:

cr = classification_report(sample_df["label"], predicted_sentiment, target_names=["Negative", "Neutral", "Positive"])
print(cr)

              precision    recall  f1-score   support

    Negative       0.53      0.59      0.56       182
     Neutral       0.50      0.78      0.61       159
    Positive       0.62      0.20      0.30       159

    accuracy                           0.53       500
   macro avg       0.55      0.52      0.49       500
weighted avg       0.55      0.53      0.49       500

